# 原始G.17同版半导体工业生产数值

按冻结plan从已封存的192份原始Table1/1D NAICS3344读取同一发布版本六个月季调指数。只建来源事件Qlib Recorder，不访问SOXX结果或训练。

In [1]:
from pathlib import Path
import hashlib,json,os
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from src.core.recorder_sources import load_verified_recorder_objects
from src.data.fed_g17_ip_vintages import parse_ip_supplement
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd().resolve()
while not (root/"SPEC.md").is_file():
    if root.parent==root:raise RuntimeError("workspace missing")
    root=root.parent
card=root/"research/semiconductor_ip_numeric_vintage_20261022"
plan_path=card/"plan.json";plan=json.loads(plan_path.read_text())
source=root/plan['source_card'];sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['execution_authorized']
old_state=json.loads((card/'run_state.json').read_text())
repair=json.loads((card/'seven_month_repair_amendment.json').read_text())
attempt=json.loads((card/'execution_attempt1.json').read_text())
diagnostic=json.loads((card/'diagnostic_result.json').read_text())
assert old_state['status']=='source_object_load_attempted' and old_state['source_load_attempts']==1
assert attempt['verified_source_recorder_object_loads_consumed']==1
assert diagnostic['verified_source_object_loads_total']==2 and diagnostic['failed_release_rows']==1
assert repair['original_plan_sha256']==sha(plan_path) and repair['new_total_verified_source_loads_max']==3
assert repair['diagnostic_result_sha256']==sha(card/'diagnostic_result.json')
for f,k in [('summary.json','source_summary_sha256'),('verification_manifest.json','source_manifest_sha256'),('publication_inventory.csv','source_inventory_sha256')]:assert sha(source/f)==plan[k]
assert sha(root/plan['parser'])==repair['repair_parser_sha256']
assert sha(root/'research/semiconductor_ip_original_g17_bridge_20261022/source_row_audit.json')==plan['row_presence_result_sha256']
assert sha(root/'research/semiconductor_ip_g17_layout_exception_20261022/adjudication.json')==plan['two_layout_adjudication_sha256']
atomic_json({'status':'repair_completion_source_load_attempted','plan_sha256':sha(plan_path),'source_load_attempts':3,'original_parser_sha256':plan['parser_sha256'],'repair_parser_sha256':repair['repair_parser_sha256'],'model_fits':0,'accounts':0},card/'run_state.json')
os.environ.setdefault('MLFLOW_ALLOW_FILE_STORE','true')
source_summary=json.loads((source/'summary.json').read_text());qlib.init(provider_uri=source_summary['provider_uri'],region='us',kernels=1)
objects,source_audit=load_verified_recorder_objects(source,expected_recorder_id=plan['source_recorder_id'],artifacts=[plan['source_raw_artifact']],expected_manifest_sha256=plan['source_manifest_sha256'],expected_artifact_sha256={plan['source_raw_artifact']:plan['source_raw_artifact_sha256']})
raw=objects[plan['source_raw_artifact']];inventory=pd.read_csv(source/'publication_inventory.csv')
assert len(raw)==len(inventory)==plan['expected_original_releases']==192
assert inventory.raw_sha256.is_unique and inventory.release_date.is_unique
by_sha=inventory.set_index('raw_sha256')
records=[]
for name,body in sorted(raw.items()):
    digest=hashlib.sha256(body).hexdigest();assert digest in by_sha.index
    row=by_sha.loc[digest]
    if name==repair['exception_release_file']:
        assert digest==repair['exception_raw_sha256'] and str(row.release_date)==repair['exception_release_date'] and str(row.observation_month)==repair['exception_observation_month']
        columns=7
    else:columns=6
    event=parse_ip_supplement(body,expected_date=row.release_date,observation_month=row.observation_month,monthly_columns=columns)
    if name!='Current_sup.txt':assert name[:8]==pd.Timestamp(event['release_date']).strftime('%Y%m%d')
    else:assert event['release_date']=='2026-09-18'
    event['source_filename']=name
    records.append(event)
events=pd.DataFrame(records).sort_values('release_date').reset_index(drop=True)
assert events.release_date.is_unique and len(events)==192
assert set(events.index_base_year)=={2007,2012,2017}
assert np.isfinite(events.ip_mom_pct_same_vintage).all()
for name in plan['positive_controls']:assert name in set(events.source_filename)
months=pd.PeriodIndex(pd.to_datetime(events.observation_month),freq='M')
expected=pd.period_range(start=months.min(),end=months.max(),freq='M')
missing=sorted(str(p) for p in expected.difference(months))
base_counts={str(int(k)):int(v) for k,v in events.index_base_year.value_counts().items()}
audit={'status':'original_same_vintage_numeric_rows_extracted_no_Stage1_feature','plan_sha256':sha(plan_path),'source_recorder_id':plan['source_recorder_id'],'release_events':len(events),'first_release':events.release_date.min(),'last_release':events.release_date.max(),'first_observation_month':events.observation_month.min(),'last_observation_month':events.observation_month.max(),'missing_observation_months':missing,'base_year_counts':base_counts,'blank_line_layout_count':int(events.layout_blank_lines.gt(0).sum()),'seven_month_release_count':int(events.published_months.map(len).eq(7).sum()),'NAICS_codes':sorted(events.NAICS.unique().tolist()),'all_six_months_positive':True,'same_vintage_mom_min_pct':float(events.ip_mom_pct_same_vintage.min()),'same_vintage_mom_max_pct':float(events.ip_mom_pct_same_vintage.max()),'numeric_source_events':len(events),'Stage1_aligned_feature_rows':0,'historical_first_receipt_qualified':False,'cross_base_absolute_level_comparability_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'goal_achieved':False}
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
    recorder=R.get_recorder()
    recorder.save_objects(artifact_path='inputs',plan=plan,source_audit=source_audit)
    recorder.save_objects(artifact_path='sources',vintage_events=events)
    recorder.save_objects(artifact_path='analysis',source_numeric_audit=audit)
    recorder.log_metrics(release_events=len(events),model_fits=0,native_accounts=0)
    rid,eid=recorder.id,str(recorder.experiment_id)
artifacts=tracking/eid/rid/'artifacts'
atomic_json({'recorder_id':rid,'entries':[{'path':str(p.relative_to(card)),'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json')
audit['recorder_id']=rid;audit['experiment_id']=eid
atomic_json(audit,card/'source_numeric_audit.json')
binding=json.loads((source/'nas_binding.json').read_text());binding.pop('evidence',None);binding.update(source_id=plan['experiment'],source_plan='plan.json',as_of='2026-09-18');atomic_json(binding,card/'nas_binding.json')
nas=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
summary={'experiment':plan['experiment'],'decision':'numeric_source_events_extracted_Stage1_timing_and_information_unqualified','objective_achieved':False,'source_numeric_audit':'source_numeric_audit.json','source_recorder_id':plan['source_recorder_id'],'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'recorder_local_dir':str(tracking/eid/rid),'nas_evidence':nas,'model_fits':0,'native_accounts':0,'Stage1_aligned_feature_rows':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','sources/*','analysis/*']}}
atomic_json(summary,card/'summary.json')
atomic_json({'status':'completed','plan_sha256':sha(plan_path),'source_load_attempts':3,'recorder_id':rid,'model_fits':0,'accounts':0},card/'run_state.json')
(card/'conclusion.md').write_text(f"# 原始G.17半导体工业生产同版数值提取\n\n从封存的192份美联储G.17 Table1/1D NAICS3344原件逐版提取六个月季调工业生产指数；每个发布版本用当版最新月/前一月计算增长，保留原六值与2007/2012/2017基年，不跨版拼接绝对指数。得到{len(events)}个来源事件，最早发布{audit['first_release']}、最新{audit['last_release']}；月度缺口{missing}，两份年度版式空行及一份七个月发布列均按原件解析。全部事件、月份与来源SHA在Qlib Recorder `{rid}`；NAS SHA `{nas['sha256']}`。\n\n这是来源数值，不是已对齐Stage1决策日的因子。原始网页09:15标记不认证历史首次接收；跨基年语义、修订及可用时钟仍需独立审查。0市场结果、0拟合、0账户，SOXX目标未完成。决定：`numeric_source_events_extracted_Stage1_timing_and_information_unqualified`。下一步冻结Qlib日历/可用时点与同源39项输入的匹配信息门，不用最新修订倒填历史。\n")
print(json.dumps({'decision':summary['decision'],'events':len(events),'base_years':base_counts,'missing_months':missing,'recorder_id':rid}))

{"decision": "numeric_source_events_extracted_Stage1_timing_and_information_unqualified", "events": 192, "base_years": {"2012": 70, "2017": 63, "2007": 59}, "missing_months": ["2025-10"], "recorder_id": "06714d1db51e487aa095d918a7eaca69"}


## 中文结论

原生Recorder、`conclusion.md`、`source_numeric_audit.json`与`readback_verification.json`保存实际结果。`interpretation_note.json`说明2025-10缺少独立最新月发布事件，但其指数在2025-12-23原件中出现；不倒填可用时间。来源数值不等于Stage1 PIT因子、预测性或SOXX超额。